# CinnaSense AI — Module 02 image model (MobileNetV2, Colab)

Trains the **current visible condition** classifier for stored cinnamon from the `CinnaSense_Dataset` Drive folder:

| Drive folder | Model class |
|---|---|
| `Good_quality` | `good` |
| `Suspected_Mold` | `suspected_surface_growth` |

This is task (A) only, the visible condition in one photo. It does **not** forecast future degradation and does **not** diagnose mould.
A `suspected_surface_growth` result means the image should be checked by an expert or a lab method.

**How to run**
1. *Runtime → Change runtime type → T4 GPU*.
2. In Google Drive, open the shared `CinnaSense_Dataset` folder and choose **Organise → Add shortcut → My Drive**. If the folder is already in your My Drive, skip this.
3. *Runtime → Run all*. Leave the **Final evaluation** section until you have finished choosing settings using the validation results.

Pipeline: inventory and integrity check → near-duplicate grouping → **group-safe** 70/15/15 split → frozen-backbone training → optional fine-tuning → single held-out test evaluation → exported artifacts.

In [1]:
# 1. Environment
!nvidia-smi -L || echo "No GPU detected: enable a GPU runtime for reasonable training time."
import os, sys, json, hashlib, shutil, re, random, datetime, platform
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers
import sklearn
from sklearn.metrics import (classification_report, confusion_matrix, f1_score,
                             roc_auc_score, precision_recall_fscore_support)

print("python", platform.python_version(), "| tensorflow", tf.__version__,
      "| keras", tf.keras.__version__ if hasattr(tf.keras, "__version__") else "?",
      "| sklearn", sklearn.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

"No GPU detected: enable a GPU runtime for reasonable training time."


'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


python 3.13.5 | tensorflow 2.21.0 | keras 3.15.1 | sklearn 1.6.1
GPUs: []


In [2]:
# 2. Configuration: edit here only
SEED = 42
DATASET_FOLDER_NAME = "CinnaSense_Dataset"
DRIVE_FOLDER_ID = "1i6UTeMYK5Fn8bWKIT5lBSwxOQtvfsI7b"   # fallback download if the folder is not in My Drive

# Drive folder name -> model class name. The order of this dict fixes the class index order.
CLASS_MAP = {"Good_quality": "good", "Suspected_Mold": "suspected_surface_growth"}
CLASSES = list(CLASS_MAP.values())

# Physical-sample ID from the file name (first capture group), e.g. r"^(CIN-S\d+)_".
# Leave "" when file names carry no sample ID; near-duplicate grouping is then the leakage guard.
SAMPLE_REGEX = r""
NEAR_DUP_BITS = 6          # dHash Hamming distance (of 64) treated as "same scene/quill"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 25                # frozen backbone; early stopping picks the best epoch
FINE_TUNE_EPOCHS = 0       # set to e.g. 10 after reviewing validation results
FINE_TUNE_LAYERS = 30
LEARNING_RATE = 1e-3
DROPOUT = 0.3

RUN_DATE = datetime.date.today().isoformat()
MODEL_VERSION = f"m02-mobilenetv2-{len(CLASSES)}cls-{RUN_DATE}-seed{SEED}"
LOCAL_DATA = Path("/content/data")
OUT_DIR_NAME = MODEL_VERSION

random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
print(MODEL_VERSION, CLASSES)

m02-mobilenetv2-2cls-2026-10-21-seed42 ['good', 'suspected_surface_growth']


In [ ]:
# 3. Get the dataset (Drive mount first; public-folder download as fallback)
from google.colab import drive
drive.mount("/content/drive")
MYDRIVE = Path("/content/drive/MyDrive")

candidates = [MYDRIVE / DATASET_FOLDER_NAME, *MYDRIVE.glob(f"*/{DATASET_FOLDER_NAME}")]
source = next((p for p in candidates if p.is_dir() and all((p / f).is_dir() for f in CLASS_MAP)), None)

if LOCAL_DATA.exists():
    shutil.rmtree(LOCAL_DATA)
if source:
    print("Copying from", source)
    shutil.copytree(source, LOCAL_DATA)            # local disk is much faster than Drive during training
else:
    print("Folder not found in My Drive; trying public download (add a shortcut to My Drive if this fails).")
    !pip -q install -U gdown
    !gdown --folder --remaining-ok "https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}" -O /content/gdown_data
    found = next(p for p in Path("/content/gdown_data").rglob("*") if p.is_dir() and all((p / f).is_dir() for f in CLASS_MAP))
    shutil.copytree(found, LOCAL_DATA)
    print("NOTE: gdown downloads at most 50 files per folder. If counts below look truncated, use the Drive shortcut instead.")

OUTPUT_DIR = MYDRIVE / "CinnaSense_models" / OUT_DIR_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for folder in CLASS_MAP:
    print(f"{folder:18s}", sum(1 for _ in (LOCAL_DATA / folder).rglob("*") if _.is_file()), "files")
print("Artifacts will be saved to", OUTPUT_DIR)

FileNotFoundError: Dataset folder not found: C:\Users\YourUsername\Downloads\Cinnamon_Dataset

In [ ]:
# 4. Inventory: integrity check, sha256, perceptual hash
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def dhash_bits(img):
    g = np.asarray(img.convert("L").resize((9, 8), Image.BILINEAR), dtype=np.int16)
    return (g[:, 1:] > g[:, :-1]).flatten()

rows, unreadable = [], []
for folder, label in CLASS_MAP.items():
    for path in sorted((LOCAL_DATA / folder).rglob("*")):
        if not path.is_file() or path.suffix.lower() not in EXTS:
            continue
        try:
            with Image.open(path) as im:
                im.verify()
            with Image.open(path) as im:
                im = im.convert("RGB"); w, h = im.size; bits = dhash_bits(im)
        except Exception as exc:
            unreadable.append((str(path), repr(exc))); continue
        m = re.search(SAMPLE_REGEX, path.name) if SAMPLE_REGEX else None
        rows.append({"image_id": f"IMG-{len(rows)+1:05d}", "file_path": str(path), "file_name": path.name,
                     "source_folder": folder, "label": label, "width": w, "height": h,
                     "sample_id_from_name": m.group(1) if m else None,
                     "sha256": hashlib.sha256(path.read_bytes()).hexdigest(), "_dhash": bits})
inv = pd.DataFrame(rows)
print(f"{len(inv)} readable images, {len(unreadable)} unreadable (skipped)")
for p, e in unreadable[:10]: print("  unreadable:", p, e)

# Exact duplicates: same bytes in two classes is a label conflict -> drop all copies; same class -> keep one.
conflict = inv.groupby("sha256")["label"].nunique() > 1
n_conflict = int(inv["sha256"].isin(conflict[conflict].index).sum())
inv = inv[~inv["sha256"].isin(conflict[conflict].index)]
n_before = len(inv); inv = inv.drop_duplicates("sha256").reset_index(drop=True)
print(f"Dropped {n_conflict} images whose identical copy sits in another class, and {n_before-len(inv)} same-class exact duplicates.")
display(inv.groupby("label").agg(images=("image_id", "size"), median_w=("width", "median"), median_h=("height", "median")))

In [ ]:
# 5. Group images that probably show the same physical quill / scene (leakage guard)
n = len(inv)
parent = list(range(n))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]; i = parent[i]
    return i
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb: parent[rb] = ra

if inv["sample_id_from_name"].notna().all():
    inv["group_id"] = inv["sample_id_from_name"]
    print("Grouping by sample ID parsed from file names.")
else:
    bits = np.stack(inv["_dhash"].to_numpy())
    for start in range(0, n, 512):                         # chunked pairwise Hamming distances
        d = (bits[start:start+512, None, :] != bits[None, :, :]).sum(-1)
        for i, j in zip(*np.nonzero(d <= NEAR_DUP_BITS)):
            if start + i < j: union(start + i, j)
    inv["group_id"] = [f"G{find(i):05d}" for i in range(n)]
    print(f"Near-duplicate grouping (dHash <= {NEAR_DUP_BITS} bits).")

g = inv.groupby("group_id").agg(size=("image_id", "size"), labels=("label", "nunique"))
print(f"{n} images -> {len(g)} groups; largest group {g['size'].max()} images; "
      f"{int((g['size'] > 1).sum())} groups contain near-duplicates.")
if g["size"].max() > 0.1 * n:
    print("WARNING: one group holds >10% of images. Lower NEAR_DUP_BITS (e.g. 3-4) and re-run this cell.")
mixed = g[g["labels"] > 1]
if len(mixed):
    print(f"WARNING: {len(mixed)} near-duplicate groups contain BOTH classes; check these labels:")
    display(inv[inv["group_id"].isin(mixed.index)][["group_id", "label", "file_name"]].head(30))

In [ ]:
# 6. Visual check: random images per class (look for labelling errors and background shortcuts)
fig, axes = plt.subplots(len(CLASSES), 6, figsize=(16, 3 * len(CLASSES)))
for row, label in zip(np.atleast_2d(axes), CLASSES):
    sub = inv[inv["label"] == label].sample(min(6, (inv["label"] == label).sum()), random_state=SEED)
    for ax, (_, r) in zip(row, sub.iterrows()):
        ax.imshow(Image.open(r.file_path).convert("RGB")); ax.set_title(f"{label}\n{r.file_name[:24]}", fontsize=8)
for ax in np.ravel(axes): ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# 7. Group-safe, class-stratified 70/15/15 split (no group appears in two partitions)
def group_split(df, seed, val_frac=0.15, test_frac=0.15):
    rng = np.random.default_rng(seed)
    grp = df.groupby("group_id").agg(label=("label", lambda s: s.mode().iloc[0]), size=("image_id", "size")).reset_index()
    assign = {}
    for _, cg in grp.groupby("label"):
        cg = cg.iloc[rng.permutation(len(cg))]
        total = cg["size"].sum()
        parts, t, v = {"test": [], "validation": [], "train": []}, 0, 0
        for gid, size in zip(cg["group_id"], cg["size"]):   # fill test, then validation, by image count
            if t < test_frac * total:
                parts["test"].append(gid); t += size
            elif v < val_frac * total:
                parts["validation"].append(gid); v += size
            else:
                parts["train"].append(gid)
        if not parts["train"]:                                # tiny class: never leave it without training data
            parts["train"].append((parts["validation"] or parts["test"]).pop())
        for part, gids in parts.items():
            assign.update({gid: part for gid in gids})
    return df["group_id"].map(assign)

inv["split"] = group_split(inv, SEED)
assert inv.groupby("group_id")["split"].nunique().eq(1).all(), "group leakage"
counts = inv.groupby(["split", "label"]).size().unstack(fill_value=0).reindex(["train", "validation", "test"])
counts["groups"] = inv.groupby("split")["group_id"].nunique()
display(counts)
if (counts[CLASSES] == 0).any().any():
    print("WARNING: a class is missing from some partition. Results will be unreliable; collect more independent samples.")
inv.drop(columns="_dhash").to_csv(OUTPUT_DIR / "image_inventory_with_split.csv", index=False)

In [ ]:
# 8. Input pipeline: raw RGB 0-255, resized to 224x224; preprocess_input is applied INSIDE the model (once)
CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES)}

def load_image(path, y):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE, antialias=True)
    img.set_shape((*IMG_SIZE, 3))
    return img, y

def make_ds(df, training):
    ds = tf.data.Dataset.from_tensor_slices((df["file_path"].to_numpy(), df["label"].map(CLASS_TO_ID).to_numpy().astype("int32")))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).cache()
    if training:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_df = inv[inv.split == "train"].reset_index(drop=True)
val_df = inv[inv.split == "validation"].reset_index(drop=True)
test_df = inv[inv.split == "test"].reset_index(drop=True)
train_ds, val_ds = make_ds(train_df, True), make_ds(val_df, False)
y_val = val_df["label"].map(CLASS_TO_ID).to_numpy()

counts_train = train_df["label"].value_counts()
class_weight = {CLASS_TO_ID[c]: len(train_df) / (len(CLASSES) * counts_train[c]) for c in CLASSES}
print("train/val/test images:", len(train_df), len(val_df), len(test_df), "| class weights:", class_weight)

In [ ]:
# 9. Model: ImageNet MobileNetV2, frozen backbone, GAP + dropout + softmax head
def build_model():
    base = tf.keras.applications.MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights="imagenet")
    base.trainable = False
    inputs = tf.keras.Input((*IMG_SIZE, 3), name="rgb_0_255")
    x = layers.RandomFlip("horizontal")(inputs)          # training-only augmentation
    x = layers.RandomRotation(0.05)(x)                    # no colour jitter: colour is itself a quality cue
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(DROPOUT)(x)
    outputs = layers.Dense(len(CLASSES), activation="softmax", name="class_probabilities")(x)
    return tf.keras.Model(inputs, outputs, name="m02_mobilenetv2"), base

class ValMacroF1(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        pred = self.model.predict(val_ds, verbose=0).argmax(1)
        logs["val_macro_f1"] = f1_score(y_val, pred, average="macro", labels=list(range(len(CLASSES))), zero_division=0)
        print(f" - val_macro_f1: {logs['val_macro_f1']:.4f}")

def callbacks():
    return [ValMacroF1(),   # first, so the callbacks below can monitor val_macro_f1
            tf.keras.callbacks.EarlyStopping(monitor="val_macro_f1", mode="max", patience=6, restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6)]

model, base = build_model()
model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary(show_trainable=True)

In [ ]:
# 10. Stage 1: train the head
hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, class_weight=class_weight, callbacks=callbacks())
history = {k: [float(v) for v in vals] for k, vals in hist.history.items()}

In [ ]:
# 11. Stage 2 (optional): fine-tune the top backbone layers with a 10x smaller learning rate, BatchNorm frozen
if FINE_TUNE_EPOCHS > 0:
    base.trainable = True
    for layer in base.layers[:-FINE_TUNE_LAYERS]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE / 10), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    ft = model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS, class_weight=class_weight, callbacks=callbacks())
    for k, vals in ft.history.items():
        history[k] = history.get(k, []) + [float(v) for v in vals]
else:
    print("Fine-tuning skipped (FINE_TUNE_EPOCHS = 0).")

In [ ]:
# 12. Learning curves and validation report (used for choosing settings; the test set is still untouched)
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(history["loss"], label="train"); ax[0].plot(history["val_loss"], label="validation"); ax[0].set_title("loss")
ax[1].plot(history["accuracy"], label="train"); ax[1].plot(history["val_accuracy"], label="validation"); ax[1].set_title("accuracy")
ax[2].plot(history["val_macro_f1"]); ax[2].set_title("validation macro-F1")
for a in ax: a.set_xlabel("epoch")
ax[0].legend(); ax[1].legend()
plt.show()
val_pred = model.predict(val_ds, verbose=0).argmax(1)
print(classification_report(y_val, val_pred, labels=list(range(len(CLASSES))), target_names=CLASSES, zero_division=0))

## Final evaluation: run once

Run this only after you have finished choosing settings using the validation results. If you change settings after looking at the test results, the test set stops being an independent check.
Always report scores together with the **number of independent test groups** shown below.

In [ ]:
# 13. Held-out test evaluation
test_ds = make_ds(test_df, False)
y_test = test_df["label"].map(CLASS_TO_ID).to_numpy()
probs = model.predict(test_ds, verbose=0)
pred = probs.argmax(1)
labels = list(range(len(CLASSES)))

report = classification_report(y_test, pred, labels=labels, target_names=CLASSES, output_dict=True, zero_division=0)
cm = confusion_matrix(y_test, pred, labels=labels)
onehot = np.eye(len(CLASSES))[y_test]
test_report = {
    "model_version": MODEL_VERSION,
    "n_test_images": int(len(test_df)),
    "n_test_groups": int(test_df["group_id"].nunique()),
    "macro_f1": float(f1_score(y_test, pred, average="macro", labels=labels, zero_division=0)),
    "accuracy": float((pred == y_test).mean()),
    "brier_score": float(np.mean(np.sum((probs - onehot) ** 2, axis=1))),
    "per_class": {c: report[c] for c in CLASSES},
    "confusion_matrix": cm.tolist(),
}
if len(CLASSES) == 2 and len(set(y_test)) == 2:
    test_report["roc_auc"] = float(roc_auc_score(y_test, probs[:, 1]))
print(json.dumps({k: v for k, v in test_report.items() if k not in ("per_class", "confusion_matrix")}, indent=2))
print(classification_report(y_test, pred, labels=labels, target_names=CLASSES, zero_division=0))

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
for a, mat, title in ((ax[0], cm, "Confusion matrix (counts)"),
                      (ax[1], cm / np.maximum(cm.sum(1, keepdims=True), 1), "Row-normalised (recall)")):
    a.imshow(mat, cmap="Blues"); a.set_title(title)
    a.set_xticks(labels, CLASSES, rotation=20, ha="right"); a.set_yticks(labels, CLASSES)
    a.set_xlabel("predicted"); a.set_ylabel("true")
    for i in labels:
        for j in labels:
            a.text(j, i, f"{mat[i, j]:.2f}" if mat.dtype.kind == "f" else int(mat[i, j]), ha="center", va="center")
# Reliability diagram: is the confidence trustworthy?
conf = probs.max(1); correct = (pred == y_test)
bins = np.linspace(0.5 if len(CLASSES) == 2 else 0, 1, 6)
idx = np.clip(np.digitize(conf, bins) - 1, 0, len(bins) - 2)
xs = [conf[idx == b].mean() for b in range(len(bins) - 1) if (idx == b).any()]
ys = [correct[idx == b].mean() for b in range(len(bins) - 1) if (idx == b).any()]
ax[2].plot([0, 1], [0, 1], "--", color="grey"); ax[2].plot(xs, ys, "o-")
ax[2].set_xlabel("mean confidence"); ax[2].set_ylabel("accuracy"); ax[2].set_title("Reliability (test)")
plt.tight_layout(); fig.savefig(OUTPUT_DIR / "test_evaluation.png", dpi=150); plt.show()

errors = test_df.assign(predicted=[CLASSES[i] for i in pred], confidence=conf)[~correct]
print(f"{len(errors)} misclassified test images")
if len(errors):
    fig, axes = plt.subplots(1, min(6, len(errors)), figsize=(16, 3.5), squeeze=False)
    for a, (_, r) in zip(axes[0], errors.sort_values("confidence", ascending=False).iterrows()):
        a.imshow(Image.open(r.file_path).convert("RGB")); a.axis("off")
        a.set_title(f"true {r.label}\npred {r.predicted} ({r.confidence:.2f})", fontsize=8)
    plt.show()

In [ ]:
# 14. Save artifacts (model, class map, preprocessing contract, config, reports, split manifests)
model.save(str(OUTPUT_DIR / "model.keras"))
(OUTPUT_DIR / "class_indices.json").write_text(json.dumps(CLASS_TO_ID, indent=2))
(OUTPUT_DIR / "preprocessing.json").write_text(json.dumps({
    "input": "RGB, uint8 0-255 values as float32, shape (224, 224, 3), batch dimension first",
    "resize": "tf.image.resize to 224x224 with antialias=True (aspect ratio not preserved)",
    "normalisation": "mobilenet_v2.preprocess_input is inside the model; do NOT normalise again",
    "output": "softmax probabilities in class_indices order",
}, indent=2))
(OUTPUT_DIR / "config.json").write_text(json.dumps({
    "model_version": MODEL_VERSION, "seed": SEED, "class_map": CLASS_MAP, "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE, "epochs": EPOCHS, "fine_tune_epochs": FINE_TUNE_EPOCHS,
    "fine_tune_layers": FINE_TUNE_LAYERS, "learning_rate": LEARNING_RATE, "dropout": DROPOUT,
    "class_weight": {CLASSES[k]: v for k, v in class_weight.items()},
    "grouping": "sample_id_from_name" if SAMPLE_REGEX else f"dHash<= {NEAR_DUP_BITS} bits",
    "split_counts": json.loads(counts.to_json(orient="index")),
    "versions": {"python": platform.python_version(), "tensorflow": tf.__version__, "sklearn": sklearn.__version__},
}, indent=2))
(OUTPUT_DIR / "history.json").write_text(json.dumps(history, indent=2))
if "test_report" in globals():
    (OUTPUT_DIR / "test_report.json").write_text(json.dumps(test_report, indent=2))
    errors.drop(columns="_dhash", errors="ignore").to_csv(OUTPUT_DIR / "test_errors.csv", index=False)
for part, df in (("train", train_df), ("validation", val_df), ("test", test_df)):
    df.drop(columns="_dhash").to_csv(OUTPUT_DIR / f"{part}.csv", index=False)
print(sorted(p.name for p in OUTPUT_DIR.iterdir()))

In [ ]:
# 15. Reload check + inference helper (same contract the backend API will use)
reloaded = tf.keras.models.load_model(str(OUTPUT_DIR / "model.keras"))
idx_to_class = {v: k for k, v in json.loads((OUTPUT_DIR / "class_indices.json").read_text()).items()}

def predict_image(path):
    img, _ = load_image(tf.constant(str(path)), 0)
    p = reloaded.predict(img[None, ...], verbose=0)[0]
    top = int(p.argmax())
    status = ("No visible surface growth detected in this photo" if idx_to_class[top] == "good"
              else "Suspected surface growth: needs expert / laboratory confirmation")
    return {"model_version": MODEL_VERSION,
            "probabilities": {idx_to_class[i]: round(float(v), 4) for i, v in enumerate(p)},
            "predicted_class": idx_to_class[top], "confidence": round(float(p[top]), 4), "status": status}

sample_path = test_df.iloc[0].file_path if len(test_df) else val_df.iloc[0].file_path
a, b = predict_image(sample_path), predict_image(sample_path)
assert a == b, "inference is not deterministic"
print(json.dumps(a, indent=2))